#  广播机制（Broadcasting）

## 一、什么是广播？
广播 = 自动扩展张量形状以兼容运算

In [1]:
import torch

a = torch.tensor([1, 2, 3])      # shape: (3,)
b = torch.tensor([[10], [20]])   # shape: (2, 1)

c = a + b  # 能运行吗？结果是什么？
print(c)
# 输出：
# tensor([[11, 12, 13],
#         [21, 22, 23]])

tensor([[11, 12, 13],
        [21, 22, 23]])


a 被“广播”成 (2, 3)：每行都是 [1,2,3]  
b 被“广播”成 (2, 3)：每列都是 [10,20]ᵀ  
然后逐元素相加  
关键：广播不会真正复制数据，只是在计算时“虚拟扩展”，因此高效！  

## 二、广播的三条规则（必须按顺序检查）
PyTorch 按以下规则判断两个张量是否可广播：

对齐维度：从最后一个维度开始，向前对齐。  
维度兼容条件（对每个对齐的维度）：  
两者相等，或  
其中一个是 1，或  
其中一个不存在（即张量维度更少）  
结果形状：每个维度取两者中的最大值。  
记忆口诀：“后对齐，1可扩，不等就崩”  

### 示例1：向量 + 标量

In [ ]:
a = torch.tensor([1, 2, 3])  # (3,)
b = torch.tensor(10)         # () ← 标量（0维）

c = a + b  # 标量广播到所有元素
print(c)   # [11, 12, 13]
# 对齐：a 有 dim0=3，b 无维度 → b 视为 (1,)？不，标量可广播到任意形状！

tensor([11, 12, 13])


### 示例2：矩阵 + 向量（最常见！）

In [ ]:
A = torch.tensor([[1., 2., 3.],    # (2, 3)
                  [4., 5., 6.]])
b = torch.tensor([10., 20., 30.])  # (3,)

C = A + b  # b 广播到每一行
print(C)

# 对齐：A 的最后维=3，b 的最后维=3 → 相等 
# A 有2行，b 只有1维 → b 缺少前维，视为 (1, 3) → 可广播为 (2, 3)

tensor([[11., 22., 33.],
        [14., 25., 36.]])


### 示例3：列向量 + 行向量（外和）

In [4]:
col = torch.tensor([[1.], [2.]])   # (2, 1)
row = torch.tensor([10., 20.])     # (2,)

result = col + row  # (2,1) + (2,) → 先对齐为 (2,1) + (1,2) → (2,2)
print(result)

tensor([[11., 21.],
        [12., 22.]])


###  示例4：不可广播的情况

In [6]:
a = torch.randn(2, 3)  # (2,3)
b = torch.randn(3, 2)  # (3,2)

c = a + b
# c = a + b  # 报错！
# RuntimeError: The size of tensor a (3) must match ...

RuntimeError: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1

## 四、广播在深度学习中的典型应用

### 1. 偏置（Bias）加法
全连接层：output = x @ W + b

x @ W 形状：(batch, out_features)
b 形状：(out_features,)
广播：b 自动加到每个样本上

In [8]:
x = torch.randn(4, 10)
W = torch.randn(10, 5)
b = torch.randn(5)

output = x @ W + b  # (4,5) + (5,) → 广播成功
print(output)
print(output.shape)  # torch.Size([4, 5])

tensor([[ 7.2298,  1.5222, -0.5300, -1.0477,  2.7307],
        [12.8996,  3.6546,  2.5360,  1.2783,  5.3903],
        [-1.9589, -3.1282,  1.2104, -0.6174,  1.3649],
        [ 6.3838,  1.1786,  0.8716, -1.6167,  4.0039]])
torch.Size([4, 5])


### 2. Batch Normalization 中的缩放和平移
gamma 和 beta 是 (num_features,)  
输入是 (batch, num_features, ...) → 自动广播到所有位置

### 3. Attention 中的 Mask 扩展

In [16]:
# 模拟输入
batch_size = 2
num_heads = 3
seq_len = 4

# 随机生成 attention scores
attention_scores = torch.randn(batch_size, num_heads, seq_len, seq_len)

# 创建 mask: (batch, seq_len)
mask = torch.tensor([[1, 1, 1, 0],
                     [1, 1, 0, 0]])  # 注意：shape 必须是 (2, 4)

# 扩展 mask 维度
mask = mask.unsqueeze(1).unsqueeze(1)  # (2, 1, 1, 4)

# 应用掩码
attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

print("Masked attention scores shape:", attention_scores.shape)
print("Last position of first sample (should be -inf):")
print(attention_scores[0, 0, :, 3])

Masked attention scores shape: torch.Size([2, 3, 4, 4])
Last position of first sample (should be -inf):
tensor([-inf, -inf, -inf, -inf])


## 五、如何避免广播陷阱？
不要依赖隐式广播做高维操作：显式使用 .unsqueeze() 或 .expand() 更清晰。
调试时打印形状：print(x.shape) 是最好的朋友。
慎用自动求导与广播结合：某些情况下梯度传播可能不符合直觉（罕见）。
### 安全做法示例：

In [20]:
# 不推荐：依赖广播
# loss = (pred - target).mean()

# 推荐：确保形状一致（尤其在复杂模型中）
# assert pred.shape == target.shape
# loss = (pred - target).mean()

## 实践

In [21]:
# 练习1：判断以下运算是否合法，并预测结果形状
a = torch.randn(3, 1, 5)
b = torch.randn(4, 5)

# c = a + b  # 合法吗？
# 对齐：a=(3,1,5), b=(4,5) → 视为 (1,4,5)
# 维度比较：
#   dim2: 5==5 
#   dim1: 1 vs 4 → 1可扩 
#   dim0: 3 vs 1 → 1可扩 
# 结果形状：(3,4,5)

c = a + b
print("Result shape:", c.shape)  # torch.Size([3, 4, 5])

# 练习2：手动实现广播（用 expand）
x = torch.tensor([1, 2, 3])        # (3,)
y = torch.tensor([[10], [20]])     # (2,1)

# 使用 expand 显式扩展
x_exp = x.expand(2, 3)   # 复制行
y_exp = y.expand(2, 3)   # 复制列
z = x_exp + y_exp
print("Via expand:\n", z)

# 验证与直接广播结果一致
z_direct = x + y
print("Same as direct?", torch.equal(z, z_direct))

Result shape: torch.Size([3, 4, 5])
Via expand:
 tensor([[11, 12, 13],
        [21, 22, 23]])
Same as direct? True


# 总结

| 要点       | 说明                                     |
|------------|------------------------------------------|
| 方向       | 从最后一个维度开始对齐                   |
| 兼容条件   | 维度相等 或 其中一个是1 或 缺失          |
| 结果形状   | 每个维度取最大值                         |
| 高效性     | 虚拟扩展，不复制数据                     |
| 典型应用   | 偏置加法、归一化、注意力掩码             |